# P13A — Generator stream pemantauan jembatan sintetis

Notebook ini menghasilkan 24 jembatan × 24 bulan untuk latihan monitoring setelah deployment. Skor keluaran dibuat dari fungsi tetap sebagai proxy model beku; notebook ini tidak melatih estimator ML. Pada bulan 15 sebagian sensor mengalami offset; pada bulan 19 DGP menghasilkan kenaikan kejadian. Label inspeksi tersedia dua bulan sesudah prediksi. Semua aset, sinyal, dan outcome sintetis, bukan data kondisi jembatan nyata.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED=13013
rng=np.random.default_rng(SEED)
n_assets=24; n_months=24
asset_age=rng.integers(5,55,n_assets)
asset_base=rng.normal(0,.45,n_assets)
types=rng.choice(['girder','slab','truss'],n_assets,p=[.55,.30,.15])
rows=[]
for a in range(n_assets):
    latent=rng.normal(.18,.06)
    for m in range(n_months):
        rain=max(0,170+105*np.sin(2*np.pi*(m%12)/12)+rng.normal(0,38))
        temperature=27+3*np.sin(2*np.pi*((m+2)%12)/12)+rng.normal(0,1)
        traffic=np.clip(rng.normal(.62,.14),.15,1)
        latent=np.clip(latent + .004 + .000025*rain + rng.normal(0,.018),0,1)
        sensor=rng.normal(.20+0.46*latent+0.00055*rain+asset_base[a],.075)
        if m>=14 and (a%3==0): sensor += .19  # perubahan kalibrasi parsial
        coverage=np.clip(rng.normal(94,5),60,100)
        if m>=14 and (a%3==0): coverage=np.clip(coverage-8,50,100)
        # Skor model beku; tidak dilatih dari label masa depan.
        raw=-3.65+1.25*sensor+0.00075*rain+0.003*asset_age[a]+0.22*traffic
        risk=1/(1+np.exp(-raw))
        event_p=np.clip(.025+.24*latent+(.13 if m>=18 else 0),.01,.75)
        event=int(rng.random()<event_p)
        dt=pd.Timestamp('2023-01-01')+pd.DateOffset(months=m)
        rows.append({'bridge_id':f'SYN-{a+1:02d}','month':dt.strftime('%Y-%m'),
                     'month_index':m+1,'bridge_type':types[a],'bridge_age_years':asset_age[a],
                     'rain_30d_mm':rain,'temperature_C':temperature,'traffic_index':traffic,
                     'sensor_response':sensor,'sensor_coverage_pct':coverage,
                     'risk_score_frozen':risk,'observed_event_synth':event,
                     'label_available_month_index':m+1+2,'sensor_shift_phase':int(m>=14 and a%3==0),
                     'event_shift_phase':int(m>=18)})
df=pd.DataFrame(rows)
assert len(df)==n_assets*n_months
assert not df.duplicated(['bridge_id','month']).any()


In [2]:
out=Path('praktikum/bab13'); out.mkdir(parents=True,exist_ok=True)
df.to_csv(out/'bridge_monitor_stream_synthetic.csv',index=False)
# Orange table: identifiers/month are metas, outcome is discrete class, phase flags remain features.
cols=list(df.columns)
roles=['meta' if c in ['bridge_id','month'] else ('class' if c=='observed_event_synth' else '') for c in cols]
typespec=['string' if c in ['bridge_id','month'] else ('discrete' if c in ['bridge_type','observed_event_synth'] else 'continuous') for c in cols]
with (out/'bridge_monitor_orange.tab').open('w',encoding='utf-8',newline='') as f:
    f.write('\t'.join(cols)+'\n'); f.write('\t'.join(typespec)+'\n'); f.write('\t'.join(roles)+'\n')
    df.to_csv(f,sep='\t',index=False,header=False,na_rep='?')
print('rows:',len(df),'assets:',df.bridge_id.nunique(),'months:',df.month.nunique(),'seed:',SEED)
print('sensor-shift rows:',int(df.sensor_shift_phase.sum()),'event-shift rows:',int(df.event_shift_phase.sum()))
print('event rate:',round(df.observed_event_synth.mean(),3),'label delay: 2 months')
print(df.groupby('month_index').agg(sensor=('sensor_response','mean'),risk=('risk_score_frozen','mean'),event_rate=('observed_event_synth','mean')).tail(8).round(3))


rows: 576 assets: 24 months: 24 seed: 13013
sensor-shift rows: 80 event-shift rows: 144
event rate: 0.142 label delay: 2 months
             sensor   risk  event_rate
month_index                           
17            0.679  0.097       0.042
18            0.649  0.091       0.000
19            0.613  0.087       0.250
20            0.585  0.080       0.125
21            0.578  0.075       0.333
22            0.524  0.072       0.375
23            0.596  0.079       0.375
24            0.606  0.082       0.292
